# BirdCLEF 2026 Training v35 — Transformer Head (from scratch)

## Strategy
Train a **PerchTransformer** head (TransformerEncoder, 2 layers, 8 heads)
on the same Perch embeddings from scratch. No v30 warm-start — different
architecture means weights are incompatible. Diversity with v30 GRU comes
from fundamentally different temporal modelling (attention vs. recurrence).

## Architecture differences vs v30 GRU
| | v30 GRU | v35 Transformer |
|---|---|---|
| Temporal model | BiGRU | TransformerEncoder |
| Context | sequential (causal bias) | full self-attention |
| Params | ~7M | ~6M |
| Long-range birds | limited | strong |

## Required Kaggle inputs
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-embs-v3`

## Output
Upload as `chiragggg/birdclef-2026-perch-weights-v35-transformer`
Files: `perch_tf_v35_fold0.pt` ... `perch_tf_v35_fold4.pt`


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

CFG = dict(
    folds           = 5,
    epochs          = 25,          # train from scratch — same as v23
    warmup_epochs   = 2,
    lr              = 5e-4,        # same as v23 (fresh training)
    batch_size      = 4,
    num_workers     = 2,
    seed            = 42,
    perch_emb_dim   = 1536,
    perch_emb_noise = 0.02,        # slightly more noise for fresh training
    tf_d_model      = 512,         # Transformer d_model
    tf_nhead        = 8,           # attention heads
    tf_layers       = 2,           # TransformerEncoder layers
    tf_ffn_dim      = 1024,        # feedforward dim inside each layer
    tf_dropout      = 0.1,
    max_seq_len     = 24,
    checkpoint_tag  = 'v35',
    device          = 'cuda' if torch.cuda.is_available() else 'cpu',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print(f"v35 Transformer Head — training from scratch")
print(f"  Device   : {device}")
print(f"  Epochs   : {CFG['epochs']}  LR={CFG['lr']}")
print(f"  Arch     : d_model={CFG['tf_d_model']}, nhead={CFG['tf_nhead']}, layers={CFG['tf_layers']}, ffn={CFG['tf_ffn_dim']}")


In [ ]:
# === CELL 2: PATHS & SPECIES ===
def _fe(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3',
)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

_all_emb = list(Path(EMBD_DIR).glob('soundscape_*.npy')) if os.path.isdir(EMBD_DIR) else []
print(f"Species            : {n_classes}")
print(f"EMBD_DIR           : {EMBD_DIR}")
print(f"  soundscape .npy  : {len(_all_emb)}")
print("Training from scratch — no pre-trained GRU checkpoint needed.")


In [ ]:
# === CELL 3: LABEL HELPERS ===
def soundscape_to_multihot(label_str):
    '''Convert semicolon-separated taxon-ID string to multi-hot vector.'''
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    '''HH:MM:SS -> total seconds.'''
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

print('Label helpers defined')

In [ ]:
# === CELL 4: PERCHTRANSFORMER MODEL ===
import math

class PerchTransformer(nn.Module):
    """TransformerEncoder head over Perch 1536-d embeddings.
    Provides architectural diversity vs v30 BiGRU — full self-attention
    can model long-range bird co-occurrence within a soundscape.
    Input : (B, T, 1536)
    Output: (B, T, n_classes)
    """
    def __init__(self, n_classes, emb_dim=1536, d_model=512, nhead=8,
                 num_layers=2, ffn_dim=1024, dropout=0.1):
        super().__init__()
        # Project Perch 1536 -> d_model
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, d_model),
            nn.GELU(),
        )
        # Learnable positional encoding (up to max_seq_len positions)
        self.pos_emb = nn.Embedding(CFG['max_seq_len'] + 1, d_model)
        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True,  # pre-norm for stability
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.2),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x, src_key_padding_mask=None):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        B, T, _ = x.shape
        z = self.proj(x)
        # Add positional encoding
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        z   = z + self.pos_emb(pos)
        h   = self.transformer(z, src_key_padding_mask=src_key_padding_mask)
        out = self.head(h)
        return out.squeeze(1) if single else out


# Quick shape check
_m = PerchTransformer(n_classes, d_model=CFG['tf_d_model'], nhead=CFG['tf_nhead'],
                      num_layers=CFG['tf_layers'], ffn_dim=CFG['tf_ffn_dim'],
                      dropout=CFG['tf_dropout']).to(device)
_x = torch.randn(2, 8, 1536).to(device)
assert _m(_x).shape == (2, 8, n_classes), "Shape mismatch!"
del _m, _x
print(f"PerchTransformer OK  (d_model={CFG['tf_d_model']}, nhead={CFG['tf_nhead']}, layers={CFG['tf_layers']})")


In [ ]:
# === CELL 5: SOUNDSCAPE SEQUENCE DATASET ===

class SoundscapeSeqDataset(Dataset):
    '''Each item is one soundscape: all windows as sequence (T, 1536).
    Windows are sorted by end_secs for temporal order.
    '''
    def __init__(self, seq_groups, emb_root, train=True):
        self.groups   = seq_groups
        self.emb_root = Path(emb_root)
        self.train    = train

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            ep = self.emb_root / (stem + '.npy')
            if ep.exists():
                e = np.load(str(ep)).astype('float32')
                if self.train and random.random() < 0.5:
                    e += np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
                embs[t] = e
            labels[t] = lv
        x = torch.from_numpy(embs)    # (T, 1536)
        y = torch.from_numpy(labels)  # (T, n_classes)
        return x, y


def seq_collate(batch):
    '''Pad variable-length sequences; return (x_pad, y_pad, mask).'''
    xs, ys = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    return x_pad, y_pad, mask


print('SoundscapeSeqDataset + seq_collate defined')

In [ ]:
# === CELL 6: BUILD SOUNDSCAPE SEQUENCE GROUPS ===
# Parse train_soundscapes_labels.csv -> {(sc_stem, end_secs): label_vector}
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    lv       = soundscape_to_multihot(row['primary_label'])
    _sc_label_map[(sc_stem, end_secs)] = lv

# Group soundscape embedding files by soundscape stem.
# Naming convention: soundscape_{sc_stem}_{end_secs}s.npy
_sc_groups      = defaultdict(list)
_missing_labels = 0

for f in Path(EMBD_DIR).glob('soundscape_*.npy'):
    try:
        stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError:
        continue
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]   # strip 'soundscape_' prefix
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None:
        _missing_labels += 1
        continue
    _sc_groups[sc_stem].append((f.stem, end_secs, lv))

seq_groups = [
    {'stem': sc_stem, 'windows': windows}
    for sc_stem, windows in _sc_groups.items()
    if windows
]

print(f"Soundscape sequences : {len(seq_groups)}")
print(f"Total windows        : {sum(len(g['windows']) for g in seq_groups)}")
print(f"Missing labels       : {_missing_labels}")

if len(seq_groups) == 0:
    raise RuntimeError(
        f"No soundscape sequences found.\n"
        f"Check that EMBD_DIR contains soundscape_*.npy files.\n"
        f"EMBD_DIR = {EMBD_DIR}"
    )

# Sanity check: show first sequence
_g  = seq_groups[0]
_w  = sorted(_g['windows'], key=lambda w: w[1])[0]
_sp = [species[j] for j in np.where(_w[2] > 0)[0]]
print(f"Sample: {_g['stem']}  first window end={_w[1]}s  active={_sp[:5]}")

In [ ]:
# === CELL 7: 5-FOLD TRAINING FROM SCRATCH ===
# Each fold trains an independent PerchTransformer on all 66 soundscapes.
# No checkpoint warm-start — architecture is different from GRU.

print('=' * 65)
print(f"v35 Transformer  {CFG['folds']} folds  AMP={torch.cuda.is_available()}")
print(f"LR={CFG['lr']}  Epochs={CFG['epochs']}  Batch={CFG['batch_size']}")
print('=' * 65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')

sc_ds = SoundscapeSeqDataset(seq_groups, EMBD_DIR, train=True)
sc_dl = DataLoader(
    sc_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers'],
    collate_fn=seq_collate,
    drop_last=False,
    pin_memory=_use_amp,
)
print(f"DataLoader: {len(sc_ds)} soundscapes  {len(sc_dl)} batches/epoch")

fold_results = []

for fold_idx in range(CFG['folds']):
    print(f"\nFold {fold_idx + 1}/{CFG['folds']}  training from scratch")

    model = PerchTransformer(
        n_classes, emb_dim=CFG['perch_emb_dim'],
        d_model=CFG['tf_d_model'], nhead=CFG['tf_nhead'],
        num_layers=CFG['tf_layers'], ffn_dim=CFG['tf_ffn_dim'],
        dropout=CFG['tf_dropout'],
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)

    warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                            total_iters=CFG['warmup_epochs'])
    cosine_sched = CosineAnnealingLR(
        optimizer,
        T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']),
        eta_min=1e-6,
    )
    scheduler = SequentialLR(optimizer,
                             schedulers=[warmup_sched, cosine_sched],
                             milestones=[CFG['warmup_epochs']])

    best_loss  = float('inf')
    best_state = None

    for epoch in range(CFG['epochs']):
        model.train()
        ep_loss   = 0.0
        n_batches = 0

        for x_pad, y_pad, mask in tqdm(sc_dl, desc=f"  Ep {epoch + 1}", leave=False):
            x_pad = x_pad.to(device)
            y_pad = y_pad.to(device)
            mask  = mask.to(device)

            # Transformer expects padding_mask: True = IGNORE (opposite of our mask)
            pad_mask = ~mask  # (B, T)

            optimizer.zero_grad()
            with autocast(enabled=_use_amp):
                logits = model(x_pad, src_key_padding_mask=pad_mask)  # (B, T, C)
                loss_e = _criterion(logits, y_pad)                     # (B, T, C)
                m      = mask.unsqueeze(-1).float()                    # (B, T, 1)
                loss   = (loss_e * m).sum() / m.sum().clamp(min=1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ep_loss   += loss.item()
            n_batches += 1

        ep_loss /= max(n_batches, 1)
        scheduler.step()

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Ep {epoch + 1:2d}/{CFG['epochs']}  loss={ep_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    out_ckpt = os.path.join(OUT_DIR, f"perch_tf_v35_fold{fold_idx}.pt")
    torch.save(model.state_dict(), out_ckpt)
    fold_results.append(best_loss)
    print(f"  Saved {out_ckpt}  best_loss={best_loss:.4f}")

    del model, optimizer, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTraining complete.")
print(f"Saved: {sorted([f for f in os.listdir(OUT_DIR) if 'v35' in f])}")
print(f"Best losses per fold: {['%.4f' % l for l in fold_results]}")


In [ ]:
# === CELL 8: UPLOAD AS birdclef-2026-perch-weights-v35-transformer ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-weights-v35-transformer'

_upload_dir = '/kaggle/working/upload_v35'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for pt in Path(OUT_DIR).glob('perch_tf_v35_fold*.pt'):
    dst = os.path.join(_upload_dir, pt.name)
    shutil.copy2(str(pt), dst)
    _copied.append(pt.name)
print(f"Files to upload: {sorted(_copied)}")

if not _copied:
    print("ERROR: no v35 checkpoints found in", OUT_DIR)
else:
    _meta = {
        'title':    DATASET_SLUG,
        'id':       f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
    }
    with open(os.path.join(_upload_dir, 'dataset-metadata.json'), 'w') as _mf:
        _json.dump(_meta, _mf, indent=2)

    _result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', _upload_dir, '--dir-mode', 'zip'],
        capture_output=True, text=True,
    )
    print(_result.stdout)
    if _result.returncode != 0:
        print('STDERR:', _result.stderr)
        print('If dataset already exists, run:')
        print(f'  kaggle datasets version -p {_upload_dir} -m "v35 transformer head from scratch"')
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
        print('Use in inference v30: attach as birdclef-2026-perch-weights-v35-transformer')
        print('Change checkpoint pattern to: perch_tf_v35_fold{i}.pt')